# Cleaning Crime Data

In [ ]:
import pandas as pd
import numpy as np
import statistics 
import openpyxl #Make sure to also have this downloaded so that excel can be read

Key Tasks:
- Focus on the 2025 data
- Group by location (ITl1 regions and ITl2 regions)
- Remove Wales data
- Remove national police forces / create a new data field for that

### Fininding all the unique police forces

In [ ]:
df_2025 = pd.read_excel("old_historical_crime_data.xlsx",
              sheet_name="2024_25",
              index_col=0)

# Here I am working out which are the unique locations so I can map to the ITL1 and ITL2 regions
df_copy = df_2025.copy()

df_copy.duplicated(subset='Force Name')
df_copy.duplicated(subset='Force Name').sum()
df_copy.drop_duplicates(subset='Force Name', inplace=True) # Want to know all of the unique force names so that I can find out which regions they belong to
print(df_copy["Force Name"]) # Prints all Police names so I can find out which region they're in. 


Financial Year
2024/25           Avon and Somerset
2024/25                Bedfordshire
2024/25    British Transport Police
2024/25              Cambridgeshire
2024/25                    Cheshire
2024/25                   Cleveland
2024/25                     Cumbria
2024/25                  Derbyshire
2024/25          Devon and Cornwall
2024/25                      Dorset
2024/25                      Durham
2024/25                 Dyfed-Powys
2024/25                       Essex
2024/25             Gloucestershire
2024/25          Greater Manchester
2024/25                       Gwent
2024/25                   Hampshire
2024/25               Hertfordshire
2024/25                  Humberside
2024/25                        Kent
2024/25                  Lancashire
2024/25              Leicestershire
2024/25                Lincolnshire
2024/25             London, City of
2024/25                  Merseyside
2024/25         Metropolitan Police
2024/25                     Norfolk
2024/25      

In [5]:
# Need to clean the data from Force Name
ITL_1_matches = { # Note I don't have any Scotland or Northern Ireland data due to the various devolved powers. I have mapped the Welsh and national english forces but will remove these at a later stage, so that a deep dive of english regional data only can be done.
    "Avon and Somerset": "South West (England)",
    "Bedfordshire": "East (England)",
    "British Transport Police": "UK",
    "Cambridgeshire": "East (England)",
    "Cheshire": "North West (England)",
    "Cleveland": "North East (England)",
    "Cumbria": "North West (England)",
    "Derbyshire": "East Midlands",
    "Devon and Cornwall": "South West (England)",
    "Dorset": "South West (England)",
    "Durham": "North East (England)",
    "Dyfed-Powys": "Wales",
    "Essex": "East (England)",
    "Gloucestershire": "South West (England)",
    "Greater Manchester": "North West (England)",
    "Gwent": "Wales",
    "Hampshire": "South East (England)",
    "Hertfordshire": "East (England)",
    "Humberside": "Yorkshire and The Humber",
    "Kent": "South East (England)",
    "Lancashire": "North West (England)",
    "Leicestershire": "East Midlands",
    "Lincolnshire": "East Midlands",
    "London, City of": "London",
    "Merseyside": "North West (England)",
    "Metropolitan Police": "London",
    "Norfolk": "East (England)",
    "North Wales": "Wales",
    "North Yorkshire": "Yorkshire and The Humber",
    "Northamptonshire": "East Midlands",
    "Northumbria": "North East (England)",
    "Nottinghamshire": "East Midlands",
    "South Wales": "Wales",
    "South Yorkshire": "Yorkshire and The Humber",
    "Staffordshire": "West Midlands",
    "Suffolk": "East (England)",
    "Surrey": "South West (England)",
    "Sussex": "South East (England)",
    "Thames Valley": "South East (England)",
    "Warwickshire": "West Midlands",
    "West Midlands": "West Midlands",
    "West Mercia": "West Midlands",
    "West Yorkshire": "Yorkshire and The Humber",
    "Wiltshire": "South West (England)",
    "Action Fraud": "UK",
    "CIFAS": "UK",
    "UK Finance": "UK"
    }

df_2025["ITL1 region"] = df_2025["Force Name"].map(ITL_1_matches) # Mapping regions to the various police forces

print(df_2025["ITL1 region"].isnull().sum()) # To test every police force has been mapped to make sure there are no null values
print(df_2025["ITL1 region"].info()) # To test every police force has been mapped by counting the number of rows in this column vs the document





0
<class 'pandas.Series'>
Index: 25356 entries, 2024/25 to 2024/25
Series name: ITL1 region
Non-Null Count  Dtype
--------------  -----
25356 non-null  str  
dtypes: str(1)
memory usage: 1.4+ MB
None


### Removing Wales from the data set

In [ ]:
# Our other data doesn't have Wales in it and as there is no Scotland or Northern Irish data it is best to stick to England. Therefore I have deleted Wales from the dataset

updated_df = df_2025.copy() #Making a copy which is where I remove a lot of data

welsh_forces = updated_df[updated_df["ITL1 region"] == "Wales"] # Filtering to only include Wales forces

# Can be removed but an example of filtering data to only Welsh forces
welsh_forces.drop_duplicates(subset='ITL1 region', inplace=True) # Removing all with Wales
welsh_forces.reset_index(inplace=True) # Setting the index to numbers so its easy to remove
welsh_forces.drop(0, inplace=True) # Now I have no welsh forces in this filter
# updated_df.info()


updated_df.loc[updated_df["ITL1 region"] == "Wales", "ITL1 region"] = None # Making this null values so I can remove the data
updated_df.dropna(inplace=True) # Removing Welsh data
print(updated_df.info()) # Checking to make sure it has been removed



<class 'pandas.DataFrame'>
Index: 23052 entries, 2024/25 to 2024/25
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Financial Quarter    23052 non-null  int64
 1   Force Name           23052 non-null  str  
 2   Offence Description  23052 non-null  str  
 3   Offence Group        23052 non-null  str  
 4   Offence Subgroup     23052 non-null  str  
 5   Offence Code         23052 non-null  str  
 6   Number of Offences   23052 non-null  int64
 7   ITL1 region          23052 non-null  str  
dtypes: int64(2), str(6)
memory usage: 1.6+ MB
None


### Adding in ITL2 regional data

In [19]:
ITL_2_matches = {
    "Avon and Somerset": "West of England;North Somerset, Somerset and Dorset",
    "Bedfordshire": "Bedfordshire and Hertfordshire",
    "British Transport Police": "UK",
    "Cambridgeshire": "Cambridgeshire and Peterborough",
    "Cheshire": "Cheshire",
    "Cleveland": "Tees Valley",
    "Cumbria": "Cumbria",
    "Derbyshire": "Derbyshire and Nottinghamshire",
    "Devon and Cornwall": "Devon;Cornwall and Isles of Scilly",
    "Dorset": "North Somerset, Somerset and Dorset",
    "Durham": "Northumberland, Durham and Tyne & Wear",
    "Dyfed-Powys": "Mid and South West Wales",
    "Essex": "Essex",
    "Gloucestershire": "Gloucestershire and Wiltshire",
    "Greater Manchester": "Greater Manchester",
    "Gwent": "South East Wales",
    "Hampshire": "Hampshire and Isle of Wight",
    "Hertfordshire": "Bedfordshire and Hertfordshire",
    "Humberside": "East Yorkshire and Northern Lincolnshire",
    "Kent": "Kent",
    "Lancashire": "Lancashire",
    "Leicestershire": "Leicestershire, Rutland and Northamptonshire",
    "Lincolnshire": "Lincolnshire",
    "London, City of": "Inner London - West",
    "Merseyside": "Merseyside",
    "Metropolitan Police": "Inner London - West;Inner London - East;Outer London - East and North East;Outer London - South;Outer London - West and North West",
    "Norfolk": "Norfolk",
    "North Wales": "North Wales",
    "North Yorkshire": "North Yorkshire",
    "Northamptonshire": "Leicestershire, Rutland and Northamptonshire",
    "Northumbria": "Northumberland, Durham and Tyne & Wear",
    "Nottinghamshire": "Derbyshire and Nottinghamshire",
    "South Wales": "South East Wales;Mid and South West Wales",
    "South Yorkshire": "South Yorkshire",
    "Staffordshire": "Shropshire and Staffordshire",
    "Suffolk": "Suffolk",
    "Surrey": "Surrey, East and West Sussex",
    "Sussex": "Surrey, East and West Sussex",
    "Thames Valley": "Berkshire, Buckinghamshire and Oxfordshire",
    "Warwickshire": "Herefordshire, Worcestershire and Warwickshire",
    "West Mercia": "Shropshire and Staffordshire;Herefordshire, Worcestershire and Warwickshire",
    "West Midlands": "West Midlands",
    "West Yorkshire": "West Yorkshire",
    "Wiltshire": "Gloucestershire and Wiltshire",
    "Action Fraud": "UK",
    "CIFAS": "UK",
    "UK Finance": "UK",
}

updated_df["ITL2 region"] = updated_df["Force Name"].map(ITL_2_matches) # Mapping the ITL2 regions if we would like them
# print(updated_df.head(3)) # Checking to make sure it ran correctly
# updated_df["ITL2 region"] = updated_df["ITL2 region"].apply(lambda x: x.split(";"))
print(updated_df.info())

updated_df.to_csv("CLEANED-regional_forces_crime_data_2024-2025.csv")


<class 'pandas.DataFrame'>
RangeIndex: 22464 entries, 0 to 22463
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Financial Year       22464 non-null  str  
 1   Financial Quarter    22464 non-null  int64
 2   Force Name           22464 non-null  str  
 3   Offence Description  22464 non-null  str  
 4   Offence Group        22464 non-null  str  
 5   Offence Subgroup     22464 non-null  str  
 6   Offence Code         22464 non-null  str  
 7   Number of Offences   22464 non-null  int64
 8   ITL1 region          22464 non-null  str  
 9   ITL2 region          22464 non-null  str  
dtypes: int64(2), str(8)
memory usage: 1.7 MB
None


### Removing national police forces from this data set and creating a new data frame with the national picture (Potentially to be compared to wellness data)

In [13]:

national_forces = updated_df[updated_df["ITL1 region"] == "UK"]
national_forces.reset_index(inplace=True)
# national_forces.to_csv("CLEANED-national_forces_crime_data_2024-2025.csv") # Creating new data set

print(updated_df.info())
# Removing from regional data set
updated_df.loc[updated_df["ITL1 region"] == "UK", "ITL1 region"] = None # Making this null values so I can remove the data
updated_df.dropna(inplace=True) # Removing UK data
print(updated_df.info()) # Checking to make sure it has removed
print(updated_df[updated_df["ITL1 region"] == "UK"]) # DOuble checking there are no more of these rows


<class 'pandas.DataFrame'>
Index: 23052 entries, 2024/25 to 2024/25
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Financial Quarter    23052 non-null  int64 
 1   Force Name           23052 non-null  str   
 2   Offence Description  23052 non-null  str   
 3   Offence Group        23052 non-null  str   
 4   Offence Subgroup     23052 non-null  str   
 5   Offence Code         23052 non-null  str   
 6   Number of Offences   23052 non-null  int64 
 7   ITL1 region          23052 non-null  str   
 8   ITL2 region          23052 non-null  object
dtypes: int64(2), object(1), str(6)
memory usage: 2.3+ MB
None
<class 'pandas.DataFrame'>
Index: 22464 entries, 2024/25 to 2024/25
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Financial Quarter    22464 non-null  int64 
 1   Force Name           22464 non-null  str   
 2

#### Saving new data pieces (Excel with cleaned ITL1 and 2 data - with any unnecessary columns deleted, also only one sheet instead of all & saving national picture data)
*Note to self see if there's a way I can apply this across all sheets (Though we definitely don't need them all!)

In [14]:
updated_df.reset_index(inplace=True) # Creating an index which is numbers based
national_forces.to_csv("CLEANED-national_forces_2024-2025.csv") # Saving the cleaned data